```python
from langchain_core.tools import tool, Tool, StructuredTool
```

### convert_to_openai_function

- 三要素
    - name：function name
    - description
    - parameters

In [29]:
from langchain_core.utils.function_calling import convert_to_openai_function

In [20]:
from typing import Literal
import json

In [21]:
def get_weather(city: str, unit: Literal["C", "F"] = "C") -> str:
    """Get current weather for a given city.

    Args:
        city: City name, e.g. 'Taipei'.
        unit: Temperature unit, 'C' or 'F'.
    """
    # 这里只做演示，实际可接第三方天气 API
    if city.lower() in {"taipei", "台北"}:
        return "Taipei: 26 C, sunny"
    return f"{city}: 20 {unit}, partly cloudy"

In [23]:
openai_fn = convert_to_openai_function(get_weather, strict=True)
openai_fn

{'name': 'get_weather',
 'description': 'Get current weather for a given city.',
 'parameters': {'properties': {'city': {'description': "City name, e.g. 'Taipei'.",
    'type': 'string'},
   'unit': {'default': 'C',
    'description': "Temperature unit, 'C' or 'F'.",
    'enum': ['C', 'F'],
    'type': 'string'}},
  'required': ['city'],
  'type': 'object',
  'additionalProperties': False},
 'strict': True}

In [24]:
from pydantic import BaseModel, Field

In [25]:
class GetWeatherInput(BaseModel):
    """Get current weather for a given city."""
    city: str = Field(..., description="City name, e.g. 'Taipei'")
    unit: Literal["C", "F"] = Field("C", description="Temperature unit")

In [26]:
convert_to_openai_function(GetWeatherInput)

{'name': 'GetWeatherInput',
 'description': 'Get current weather for a given city.',
 'parameters': {'properties': {'city': {'description': "City name, e.g. 'Taipei'",
    'type': 'string'},
   'unit': {'default': 'C',
    'description': 'Temperature unit',
    'enum': ['C', 'F'],
    'type': 'string'}},
  'required': ['city'],
  'type': 'object'}}

### tool, Tool, StructuredTool

- `@tool`（装饰器/工厂）
    - 把一个 Python 函数包装成 LangChain 的“工具”对象。
    - 如果函数只有一个 string 入参，通常会变成一个**Tool**（“不带结构”的单输入工具）。
    - 如果函数有多个或带类型注解的入参，会变成**StructuredTool**（带参数结构/JSON-schema 的工具）。
    - 会自动用函数名/Docstring 作为 name/description，支持同步或异步函数。
- `Tool`（类）
    - “简单工具”：单一字符串输入的场景（模型只需要给一段文本当参数）。更轻量，参数模式固定为一个字段（常见名如 tool_input 或函数唯一参数名）。适合像 search("llm agents") 这种一条字符串就够的调用。
- `StructuredTool`（类）
    - “结构化工具”：多个命名参数、需要类型/约束、或要给模型清晰 JSON Schema 的场景。会（从类型注解或自定义 args_schema）生成 Pydantic 模型 → 让函数调用式/JSON 工具调用的模型更容易正确填参。

### demo

In [17]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from langgraph.prebuilt import ToolNode


def run_queries(search_queries: list[str], **kwargs):
    """Run the generated queries."""
    return tavily_tool.batch([{"query": query} for query in search_queries])

class Reflection(BaseModel):
    missing: str = Field(description="Critique of what is missing.")
    superfluous: str = Field(description="Critique of what is superfluous")
    
class AnswerQuestion(BaseModel):
    """Answer the question. Provide an answer, reflection, and then follow up with search queries to improve the answer."""

    answer: str = Field(description="~250 word detailed answer to the question.")
    reflection: Reflection = Field(description="Your reflection on the initial answer.")
    search_queries: list[str] = Field(
        description="1-3 search queries for researching improvements to address the critique of your current answer."
    )
    
class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question. Provide an answer, reflection,

    cite your reflection with references, and finally
    add search queries to improve the answer."""

    references: list[str] = Field(
        description="Citations motivating your updated answer."
    )
    
tool_node = ToolNode(
    [
        StructuredTool.from_function(run_queries, name=AnswerQuestion.__name__),
        StructuredTool.from_function(run_queries, name=ReviseAnswer.__name__),
    ]
)

In [18]:
tool_node

tools(tags=None, recurse=True, explode_args=False, func_accepts={'config': ('N/A', <class 'inspect._empty'>), 'store': ('store', None)}, tools_by_name={'AnswerQuestion': StructuredTool(name='AnswerQuestion', description='Run the generated queries.', args_schema=<class 'langchain_core.utils.pydantic.AnswerQuestion'>, func=<function run_queries at 0x71e5182c0c10>), 'ReviseAnswer': StructuredTool(name='ReviseAnswer', description='Run the generated queries.', args_schema=<class 'langchain_core.utils.pydantic.ReviseAnswer'>, func=<function run_queries at 0x71e5182c0c10>)}, tool_to_state_args={'AnswerQuestion': {}, 'ReviseAnswer': {}}, tool_to_store_arg={'AnswerQuestion': None, 'ReviseAnswer': None}, handle_tool_errors=True, messages_key='messages')

### examples

In [1]:
from langchain_core.tools import tool, Tool, StructuredTool
from pydantic import BaseModel

In [27]:
# 1) 用 @tool 装饰器（单参 → 通常生成 Tool）
@tool
def web_search(query: str) -> str:
    "Search the web for a query."
    return f"results for: {query}"

# 2) 用 @tool 装饰器（多参 → 生成 StructuredTool）
@tool
def crop_image(path: str, x: int, y: int, w: int, h: int) -> str:
    "Crop an image given coordinates."
    return f"cropped {path} at ({x},{y},{w},{h})"

# 3) 手动构造 Tool（单字符串输入）
def summarize(text: str) -> str:
    return text[:200]

sum_tool = Tool.from_function(
    name="summarize",
    description="Summarize long text into ~200 chars.",
    func=summarize,  # or async_func=...
)

# 4) 手动构造 StructuredTool（自定义参数模式）
class ResizeInput(BaseModel):
    path: str
    width: int
    height: int

def resize_image(path: str, width: int, height: int) -> str:
    return f"resized {path} to {width}x{height}"

resize_tool = StructuredTool.from_function(
    func=resize_image,
    name="resize_image",
    description="Resize an image to given width/height.",
    args_schema=ResizeInput,  # 明确 JSON schema/校验
)

In [31]:
convert_to_openai_function(web_search)

{'name': 'web_search',
 'description': 'Search the web for a query.',
 'parameters': {'properties': {'query': {'type': 'string'}},
  'required': ['query'],
  'type': 'object'}}

In [3]:
tools = [web_search, crop_image, sum_tool, resize_tool]

In [4]:
web_search.invoke("llm agents & vector db")

'results for: llm agents & vector db'

In [5]:
crop_image.invoke({"path": "cat.png", "x": 10, "y": 20, "w": 100, "h": 80})

'cropped cat.png at (10,20,100,80)'

In [9]:
len(sum_tool.invoke("A"*500))

200

In [8]:
resize_tool.invoke({"path": "cat.png", "width": 256, "height": 256, "keep_aspect": True})

'resized cat.png to 256x256'

### agent

- https://langchain-ai.github.io/langgraph/how-tos/react-agent-from-scratch/

In [10]:
from dotenv import load_dotenv, find_dotenv

In [11]:
assert load_dotenv(find_dotenv(), override=True)

In [12]:
from langgraph.prebuilt import ToolNode

In [14]:
# 作为 react graph 的执行节点
ToolNode(tools)

tools(tags=None, recurse=True, explode_args=False, func_accepts={'config': ('N/A', <class 'inspect._empty'>), 'store': ('store', None)}, tools_by_name={'web_search': StructuredTool(name='web_search', description='Search the web for a query.', args_schema=<class 'langchain_core.utils.pydantic.web_search'>, func=<function web_search at 0x71e51956ab00>), 'crop_image': StructuredTool(name='crop_image', description='Crop an image given coordinates.', args_schema=<class 'langchain_core.utils.pydantic.crop_image'>, func=<function crop_image at 0x71e5261092d0>), 'summarize': Tool(name='summarize', description='Summarize long text into ~200 chars.', func=<function summarize at 0x71e526108040>), 'resize_image': StructuredTool(name='resize_image', description='Resize an image to given width/height.', args_schema=<class '__main__.ResizeInput'>, func=<function resize_image at 0x71e5261093f0>)}, tool_to_state_args={'web_search': {}, 'crop_image': {}, 'summarize': {}, 'resize_image': {}}, tool_to_sto